# Overlapping orders

Regardless of method, we may wind up with points from one order overlapping with
points from another. Let's try to figure out how to handle these.

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull

In [ ]:
# Taxonomy lookups.
taxonomy_list = pd.read_csv("output/cleaned_trees/Actinopterygii_genus_order_family_taxon.csv", index_col=0)
taxonomy_list.index.name = 'taxon'

In [ ]:
# Input datafile. This is the final output that is run through csv_to_openspace.py for the OpenSpace ingest.
#input_file = "output\\Actinopterygii_tSNE_relax100.csv"
#input_file = "output\\Actinopterygii_MDS_relax55.csv"
#input_file = "output\\Actinopterygii_relax100_scaffold_MDS_relax85.csv"
input_file = "output\\Actinopterygii_relax100_scaffold_tSNE_relax100.csv"
#input_file = "output\\Actinopterygii_UMAP_relax100.csv"


In [ ]:
# Load the 3D point data and join with taxonomy to get order for each point.
points_df = pd.read_csv(input_file, index_col=0)

# If the order column is missing or incomplete, fill from taxonomy_list.
if 'order' not in points_df.columns or points_df['order'].isna().any():
    points_df = points_df.join(taxonomy_list[['order']], how='left', rsuffix='_tax')
    if 'order_tax' in points_df.columns:
        points_df['order'] = points_df['order'].fillna(points_df['order_tax'])
        points_df.drop(columns=['order_tax'], inplace=True)

# Drop any rows without a valid order or coordinates.
points_df = points_df.dropna(subset=['x', 'y', 'z', 'order'])

# Remove duplicate (x, y, z) positions — keep the first occurrence.
before = len(points_df)
points_df = points_df.drop_duplicates(subset=['x', 'y', 'z'])
print(f"Removed {before - len(points_df)} duplicate points ({before} → {len(points_df)}).")

print(f"Loaded {len(points_df)} points across {points_df['order'].nunique()} orders.")
points_df.head()

In [ ]:
def build_tangent_basis(centroid: np.ndarray):
    """Return two orthonormal vectors (u1, u2) spanning the tangent plane at centroid."""
    c = centroid / np.linalg.norm(centroid)
    # Pick a seed vector not parallel to c
    seed = np.array([1.0, 0.0, 0.0]) if abs(c[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
    u1 = seed - (seed @ c) * c
    u1 /= np.linalg.norm(u1)
    u2 = np.cross(c, u1)
    u2 /= np.linalg.norm(u2)
    return u1, u2


def project_to_tangent(points: np.ndarray, u1: np.ndarray, u2: np.ndarray) -> np.ndarray:
    """Orthographic projection of unit-sphere points onto the 2D tangent plane (u1, u2)."""
    return np.column_stack([points @ u1, points @ u2])


def points_inside_hull_2d(test_pts_3d: np.ndarray, hull: ConvexHull,
                           u1: np.ndarray, u2: np.ndarray,
                           tol: float = 1e-10) -> np.ndarray:
    """Boolean mask: which of test_pts_3d lie inside the 2D convex hull?

    The hull was built in the tangent plane with basis (u1, u2).
    The hull centroid direction is u1 × u2 (orthonormal basis property).
    Points on the opposite hemisphere (dot product ≤ 0 with the centroid) are
    rejected first — orthographic projection of antipodal points creates false
    positives in the half-space test.
    """
    # Centroid direction of the hull (outward normal of the tangent plane).
    hull_centroid = np.cross(u1, u2)   # unit vector by construction

    # Gate: only consider points on the same hemisphere as the hull centroid.
    same_side = (test_pts_3d @ hull_centroid) > 0   # (n_test_points,)

    result = np.zeros(len(test_pts_3d), dtype=bool)
    if not same_side.any():
        return result

    coords_2d = project_to_tangent(test_pts_3d[same_side], u1, u2)
    A = hull.equations[:, :2]          # (n_facets, 2)
    b = hull.equations[:, 2]           # (n_facets,)
    inside = (A @ coords_2d.T + b[:, np.newaxis]) <= tol
    result[same_side] = np.all(inside, axis=0)
    return result


# Group points by order.
order_groups = {order: grp[['x', 'y', 'z']].values
                for order, grp in points_df.groupby('order')}

# Build 2D convex hulls in each order's local tangent plane.
hulls   = {}   # order -> ConvexHull
bases   = {}   # order -> (u1, u2)
skipped = []

for order, pts in order_groups.items():
    if len(pts) < 3:
        skipped.append(f"{order} (n={len(pts)}, too few points for 2D hull)")
        continue
    centroid = pts.mean(axis=0)
    centroid /= np.linalg.norm(centroid)
    u1, u2 = build_tangent_basis(centroid)
    coords_2d = project_to_tangent(pts, u1, u2)
    try:
        hulls[order] = ConvexHull(coords_2d)
        bases[order] = (u1, u2)
    except Exception as e:
        skipped.append(f"{order}: {e}")

if skipped:
    print("Skipped orders (no hull computed):")
    for s in skipped:
        print(" •", s)

print(f"\n2D tangent-plane convex hulls built for {len(hulls)} orders.")

# For each order with a hull, count points from every *other* order that fall inside.
results = {}
for hull_order, hull in hulls.items():
    u1, u2 = bases[hull_order]
    results[hull_order] = {}
    for other_order, other_pts in order_groups.items():
        if other_order == hull_order:
            continue
        mask = points_inside_hull_2d(other_pts, hull, u1, u2)
        count = int(mask.sum())
        if count > 0:
            results[hull_order][other_order] = count

In [ ]:
# Build a summary table: one row per order that has a hull.
# Columns: order, total_intruding_points, intruding_orders (name + count pairs)
rows = []
for hull_order in sorted(results.keys()):
    intruders = results[hull_order]
    total = sum(intruders.values())
    # Comma-separated list like "Perciformes (142), Clupeiformes (3)"
    detail = ", ".join(
        f"{o} ({n})" for o, n in sorted(intruders.items(), key=lambda kv: -kv[1])
    )
    rows.append({
        'Order (hull)': hull_order,
        'Hull size (# species)': len(order_groups[hull_order]),
        'Total intruding points': total,
        'Intruding orders (count)': detail if detail else "—",
    })

summary_df = pd.DataFrame(rows).set_index('Order (hull)')

# Show only orders that actually have intruders to keep the table readable.
# Remove the filter below to see all orders.
overlap_df = summary_df[summary_df['Total intruding points'] > 0].sort_values(
    'Total intruding points', ascending=False
)

print(f"Orders with at least one intruding point: {len(overlap_df)} / {len(summary_df)}\n")
overlap_df

In [ ]:
import plotly.graph_objects as go
import plotly.colors as pc

# ── 1. Graph-colour the orders so adjacent (overlapping) hulls get different colours ──────────

# Two orders conflict if either hull contains points from the other.
conflict_adj = {order: set() for order in hulls}
for hull_order, intruders in results.items():
    for intruding_order in intruders:
        conflict_adj[hull_order].add(intruding_order)
        if intruding_order in conflict_adj:
            conflict_adj[intruding_order].add(hull_order)

def greedy_color(adj):
    """Greedy graph colouring, highest-degree-first."""
    color_map = {}
    for node in sorted(adj, key=lambda n: len(adj[n]), reverse=True):
        used = {color_map[nb] for nb in adj[node] if nb in color_map}
        for c in range(len(adj)):
            if c not in used:
                color_map[node] = c
                break
    return color_map

color_idx_map = greedy_color(conflict_adj)
n_colors = max(color_idx_map.values(), default=0) + 1
print(f"Graph colouring needs {n_colors} colours for {len(hulls)} orders.")

# Build a wide-enough palette by concatenating several qualitative sets.
palette = (pc.qualitative.Plotly + pc.qualitative.D3 +
           pc.qualitative.G10 + pc.qualitative.T10 + pc.qualitative.Alphabet)

# ── 2. Helper: ordered 3D hull-boundary vertices ────────────────────────────────────────────

def hull_boundary_3d(order):
    """Return (n, 3) array of CCW hull-boundary vertices normalised to unit sphere."""
    pts_3d = order_groups[order]
    v_idx  = hulls[order].vertices          # CCW-ordered indices for 2-D hull
    verts  = pts_3d[v_idx].copy()
    verts /= np.linalg.norm(verts, axis=1, keepdims=True)
    return verts

# ── 3. Build figure ─────────────────────────────────────────────────────────────────────────

fig = go.Figure()

# Faint unit-sphere background surface
_u = np.linspace(0, 2 * np.pi, 60)
_v = np.linspace(0, np.pi, 40)
fig.add_trace(go.Surface(
    x=np.outer(np.cos(_u), np.sin(_v)),
    y=np.outer(np.sin(_u), np.sin(_v)),
    z=np.outer(np.ones_like(_u), np.cos(_v)),
    colorscale=[[0, 'rgb(220,230,245)'], [1, 'rgb(220,230,245)']],
    opacity=0.10,
    showscale=False,
    hoverinfo='skip',
    showlegend=False,
))

# Species scatter points (small, per-order colour, semi-transparent)
for order, pts_3d in order_groups.items():
    cidx  = color_idx_map.get(order, 0)
    color = palette[cidx % len(palette)]
    hover_labels = [f"{t}<br>{order}"
                    for t in points_df[points_df['order'] == order].index]
    fig.add_trace(go.Scatter3d(
        x=pts_3d[:, 0], y=pts_3d[:, 1], z=pts_3d[:, 2],
        mode='markers',
        marker=dict(size=2, color=color, opacity=0.45),
        name=order,
        legendgroup=order,
        showlegend=False,
        hovertext=hover_labels,
        hoverinfo='text',
    ))

# Filled convex hull mesh + outline for each order
for order, hull in hulls.items():
    verts = hull_boundary_3d(order)   # (n, 3)
    n     = len(verts)
    cidx  = color_idx_map.get(order, 0)
    color = palette[cidx % len(palette)]

    # Triangle fan from vertex 0 covers the full convex polygon
    i_tri = [0] * (n - 2)
    j_tri = list(range(1, n - 1))
    k_tri = list(range(2, n))

    total_intruding = sum(results.get(order, {}).values())
    hover_text = (f"{order}<br>{len(order_groups[order])} species<br>"
                  f"{total_intruding} intruding points")

    fig.add_trace(go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=i_tri, j=j_tri, k=k_tri,
        color=color,
        opacity=0.35,
        name=order,
        legendgroup=order,
        showlegend=True,
        hoverinfo='text',
        text=hover_text,
    ))

    # Closed outline around the hull
    outline = np.vstack([verts, verts[0]])
    fig.add_trace(go.Scatter3d(
        x=outline[:, 0], y=outline[:, 1], z=outline[:, 2],
        mode='lines',
        line=dict(color=color, width=2),
        name=order,
        legendgroup=order,
        showlegend=False,
        hoverinfo='skip',
    ))

# Order labels at the centroid of each hull (mean of boundary vertices, re-normalised)
label_x, label_y, label_z, label_text, label_colors = [], [], [], [], []
for order, hull in hulls.items():
    verts = hull_boundary_3d(order)
    cx, cy, cz = verts.mean(axis=0)
    # Push the label slightly outside the sphere so it sits on top of the mesh
    norm = np.sqrt(cx**2 + cy**2 + cz**2)
    scale = 1.04 / norm
    label_x.append(cx * scale)
    label_y.append(cy * scale)
    label_z.append(cz * scale)
    label_text.append(order)
    label_colors.append(palette[color_idx_map.get(order, 0) % len(palette)])

fig.add_trace(go.Scatter3d(
    x=label_x, y=label_y, z=label_z,
    mode='text',
    text=label_text,
    textfont=dict(size=9, color=label_colors),
    hoverinfo='skip',
    showlegend=False,
))

fig.update_layout(
    title=f"Order convex hulls on unit sphere — {n_colors}-colour graph colouring",
    width=950,
    height=850,
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode='cube',
    ),
    legend=dict(
        itemsizing='constant',
        title='Order',
        tracegroupgap=0,
    ),
    margin=dict(l=0, r=150, t=40, b=0),
)
fig.show()

In [ ]:
from scipy.optimize import minimize

# ── 1. Current centroids and angular "radii" for each order with a hull ────────────────────
order_list = sorted(hulls.keys())
n_orders   = len(order_list)

current_centroids = np.array([order_groups[o].mean(axis=0) for o in order_list])
current_centroids /= np.linalg.norm(current_centroids, axis=1, keepdims=True)

# Angular radius = max angular distance from the centroid to any hull-boundary vertex.
order_radii = np.array([
    np.arccos(np.clip(hull_boundary_3d(o) @ current_centroids[i], -1, 1)).max()
    for i, o in enumerate(order_list)
])

print(f"Optimising positions for {n_orders} orders on the unit sphere.")
print(f"Angular hull radii — min: {np.degrees(order_radii.min()):.1f}°  "
      f"median: {np.degrees(np.median(order_radii)):.1f}°  "
      f"max: {np.degrees(order_radii.max()):.1f}°")

# ── 2. Parameterise each centroid as (theta, phi) in spherical coordinates ────────────────
def xyz_to_spherical(xyz):
    thetas = np.arccos(np.clip(xyz[:, 2], -1, 1))
    phis   = np.arctan2(xyz[:, 1], xyz[:, 0])
    return np.concatenate([thetas, phis])

def spherical_to_xyz(tp):
    th, ph = tp[:n_orders], tp[n_orders:]
    return np.column_stack([np.sin(th) * np.cos(ph),
                             np.sin(th) * np.sin(ph),
                             np.cos(th)])

# ── 3. Energy: sum of squared penetration depths over all hull pairs ──────────────────────
def energy(tp):
    C = spherical_to_xyz(tp)
    E = 0.0
    for i in range(n_orders):
        for j in range(i + 1, n_orders):
            dist = np.arccos(np.clip(C[i] @ C[j], -1, 1))
            gap  = order_radii[i] + order_radii[j] - dist
            if gap > 0:
                E += gap ** 2
    return E

tp0 = xyz_to_spherical(current_centroids)
E0  = energy(tp0)
print(f"\nInitial energy (sum of squared angular overlaps): {E0:.6f}")

result = minimize(energy, tp0, method='L-BFGS-B',
                  options=dict(maxiter=5000, ftol=1e-12, gtol=1e-8))

Ef = result.fun
print(f"Final   energy: {Ef:.6f}  (reduction: {100 * (E0 - Ef) / E0 if E0 > 0 else 0:.1f}%)")
print(f"Converged: {result.success} — {result.message}")

# ── 4. Rotate each order's points by the rotation old_centroid → new_centroid ────────────
new_centroids = spherical_to_xyz(result.x)

def rotate_cluster(pts, old_c, new_c):
    """Rigidly rotate pts by the rotation that takes old_c to new_c (Rodrigues formula)."""
    axis  = np.cross(old_c, new_c)
    sin_a = np.linalg.norm(axis)
    cos_a = float(np.clip(old_c @ new_c, -1, 1))
    if sin_a < 1e-12:
        return pts.copy()
    axis /= sin_a
    K = np.array([[     0, -axis[2],  axis[1]],
                  [ axis[2],      0, -axis[0]],
                  [-axis[1],  axis[0],      0]])
    R = cos_a * np.eye(3) + sin_a * K + (1 - cos_a) * np.outer(axis, axis)
    rotated = (R @ pts.T).T
    rotated /= np.linalg.norm(rotated, axis=1, keepdims=True)  # guard float drift
    return rotated

# Build shifted_points_df — all orders, including those without hulls (kept in place).
shifted_rows = {}

for i, order in enumerate(order_list):
    idx     = points_df[points_df['order'] == order].index
    pts     = order_groups[order]
    shifted = rotate_cluster(pts, current_centroids[i], new_centroids[i])
    for j, taxon in enumerate(idx):
        row = points_df.loc[taxon].to_dict()
        row.update({'x': shifted[j, 0], 'y': shifted[j, 1], 'z': shifted[j, 2]})
        shifted_rows[taxon] = row

for order, pts in order_groups.items():     # orders skipped by hull builder — copy unchanged
    if order in hulls:
        continue
    for j, taxon in enumerate(points_df[points_df['order'] == order].index):
        shifted_rows[taxon] = points_df.loc[taxon].to_dict()

shifted_points_df = pd.DataFrame.from_dict(shifted_rows, orient='index')
shifted_points_df.index.name = 'taxon'

print(f"\nshifted_points_df: {len(shifted_points_df)} rows, "
      f"{shifted_points_df['order'].nunique()} orders")
shifted_points_df.head()

In [ ]:
# ── Recompute order groups, hulls, and overlap counts from shifted_points_df ──────────────

sh_order_groups = {order: grp[['x', 'y', 'z']].values
                   for order, grp in shifted_points_df.groupby('order')}

sh_hulls = {}
sh_bases = {}
sh_skipped = []
for order, pts in sh_order_groups.items():
    if len(pts) < 3:
        sh_skipped.append(order)
        continue
    c = pts.mean(axis=0)
    c /= np.linalg.norm(c)
    u1, u2 = build_tangent_basis(c)
    c2d = project_to_tangent(pts, u1, u2)
    try:
        sh_hulls[order] = ConvexHull(c2d)
        sh_bases[order] = (u1, u2)
    except Exception as e:
        sh_skipped.append(f"{order}: {e}")

sh_results = {}
for ho, hull in sh_hulls.items():
    u1, u2 = sh_bases[ho]
    sh_results[ho] = {}
    for oo, opts in sh_order_groups.items():
        if oo == ho:
            continue
        m = points_inside_hull_2d(opts, hull, u1, u2)
        if m.sum() > 0:
            sh_results[ho][oo] = int(m.sum())

total_intruding_after = sum(sum(v.values()) for v in sh_results.values())
print(f"After shift — orders with intruders: "
      f"{sum(1 for v in sh_results.values() if v)} / {len(sh_hulls)}, "
      f"total intruding points: {total_intruding_after}")

# ── Graph-colour the shifted hulls ────────────────────────────────────────────────────────
sh_conflict_adj = {order: set() for order in sh_hulls}
for ho, intruders in sh_results.items():
    for io in intruders:
        sh_conflict_adj[ho].add(io)
        if io in sh_conflict_adj:
            sh_conflict_adj[io].add(ho)

sh_color_idx_map = greedy_color(sh_conflict_adj)
sh_n_colors = max(sh_color_idx_map.values(), default=0) + 1
print(f"Graph colouring needs {sh_n_colors} colours for {len(sh_hulls)} orders.")

# ── Helper: hull boundary vertices from shifted data ──────────────────────────────────────
def sh_hull_boundary_3d(order):
    pts_3d = sh_order_groups[order]
    v_idx  = sh_hulls[order].vertices
    verts  = pts_3d[v_idx].copy()
    verts /= np.linalg.norm(verts, axis=1, keepdims=True)
    return verts

# ── Build figure ──────────────────────────────────────────────────────────────────────────
sh_fig = go.Figure()

_u = np.linspace(0, 2 * np.pi, 60)
_v = np.linspace(0, np.pi, 40)
sh_fig.add_trace(go.Surface(
    x=np.outer(np.cos(_u), np.sin(_v)),
    y=np.outer(np.sin(_u), np.sin(_v)),
    z=np.outer(np.ones_like(_u), np.cos(_v)),
    colorscale=[[0, 'rgb(220,230,245)'], [1, 'rgb(220,230,245)']],
    opacity=0.10,
    showscale=False,
    hoverinfo='skip',
    showlegend=False,
))

for order, pts_3d in sh_order_groups.items():
    cidx  = sh_color_idx_map.get(order, 0)
    color = palette[cidx % len(palette)]
    hover_labels = [f"{t}<br>{order}"
                    for t in shifted_points_df[shifted_points_df['order'] == order].index]
    sh_fig.add_trace(go.Scatter3d(
        x=pts_3d[:, 0], y=pts_3d[:, 1], z=pts_3d[:, 2],
        mode='markers',
        marker=dict(size=2, color=color, opacity=0.45),
        name=order,
        legendgroup=order,
        showlegend=False,
        hovertext=hover_labels,
        hoverinfo='text',
    ))

for order, hull in sh_hulls.items():
    verts = sh_hull_boundary_3d(order)
    n     = len(verts)
    cidx  = sh_color_idx_map.get(order, 0)
    color = palette[cidx % len(palette)]
    i_tri = [0] * (n - 2)
    j_tri = list(range(1, n - 1))
    k_tri = list(range(2, n))
    total_intruding = sum(sh_results.get(order, {}).values())
    hover_text = (f"{order}<br>{len(sh_order_groups[order])} species<br>"
                  f"{total_intruding} intruding points")
    sh_fig.add_trace(go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=i_tri, j=j_tri, k=k_tri,
        color=color, opacity=0.35,
        name=order, legendgroup=order, showlegend=True,
        hoverinfo='text', text=hover_text,
    ))
    outline = np.vstack([verts, verts[0]])
    sh_fig.add_trace(go.Scatter3d(
        x=outline[:, 0], y=outline[:, 1], z=outline[:, 2],
        mode='lines', line=dict(color=color, width=2),
        name=order, legendgroup=order, showlegend=False, hoverinfo='skip',
    ))

# Order labels
lx, ly, lz, lt, lc = [], [], [], [], []
for order in sh_hulls:
    verts = sh_hull_boundary_3d(order)
    cx, cy, cz = verts.mean(axis=0)
    norm  = np.sqrt(cx**2 + cy**2 + cz**2)
    scale = 1.04 / norm
    lx.append(cx * scale); ly.append(cy * scale); lz.append(cz * scale)
    lt.append(order)
    lc.append(palette[sh_color_idx_map.get(order, 0) % len(palette)])

sh_fig.add_trace(go.Scatter3d(
    x=lx, y=ly, z=lz,
    mode='text', text=lt,
    textfont=dict(size=9, color=lc),
    hoverinfo='skip', showlegend=False,
))

sh_fig.update_layout(
    title=f"Shifted order convex hulls — {sh_n_colors}-colour graph colouring",
    width=950, height=850,
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
               zaxis=dict(visible=False), aspectmode='cube'),
    legend=dict(itemsizing='constant', title='Order', tracegroupgap=0),
    margin=dict(l=0, r=150, t=40, b=0),
)
sh_fig.show()

In [ ]:
sh_rows = []
for hull_order in sorted(sh_results.keys()):
    intruders = sh_results[hull_order]
    total = sum(intruders.values())
    detail = ", ".join(
        f"{o} ({n})" for o, n in sorted(intruders.items(), key=lambda kv: -kv[1])
    )
    sh_rows.append({
        'Order (hull)': hull_order,
        'Hull size (# species)': len(sh_order_groups[hull_order]),
        'Total intruding points': total,
        'Intruding orders (count)': detail if detail else "—",
    })

sh_summary_df = pd.DataFrame(sh_rows).set_index('Order (hull)')
sh_overlap_df = sh_summary_df[sh_summary_df['Total intruding points'] > 0].sort_values(
    'Total intruding points', ascending=False
)

print(f"Orders with at least one intruding point: {len(sh_overlap_df)} / {len(sh_summary_df)}\n")
sh_overlap_df